In [ ]:
q1 = "I just discovered the course. Can I still join it?"
q2 = "I just found out about the program. Can I still enroll?"

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
v1 = model.encode(q1)

In [ ]:
v1.shape

In [ ]:
v2 = model.encode(q2)

In [ ]:
v2.shape

In [ ]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [ ]:
v1.dot(dv)

In [ ]:
v2.dot(dv)

In [ ]:
import sys
from pathlib import Path

AGENTS_DIR = Path.cwd().parent / "agents"
sys.path.append(str(AGENTS_DIR))

from ingest import load_faq_data

documents = load_faq_data()

In [ ]:
documents[10]

In [ ]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [ ]:
texts[10]

In [ ]:
from tqdm.auto import tqdm

In [ ]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

Hand by hand vector search

In [ ]:
import numpy as np
X = np.array(vectors)

In [ ]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [ ]:
scores = X.dot(v_query)

In [ ]:
idx = np.argmax(scores)
idx, scores[idx]

In [ ]:
documents[idx]

In [ ]:
top5 = np.argsort(scores)[-5:]

In [ ]:
top5 = top5[::-1]
top5

In [ ]:
scores

In [ ]:
scores[top5]

In [ ]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

Minsearch vector search

In [ ]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [ ]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [ ]:
results[0]

Filtrar por curso

In [ ]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

RAG Vector Search with Gemini

In [ ]:
from rag_helper import RAGBase, RAGVector

# Com Gemini
from google import genai
genai_client = genai.Client()
vector_assistant = RAGVector(embedder=model, index=vindex, llm_client=genai_client)

vector_assistant.rag("the program has already begun, can I still sign up?")